# Orfipy ORF Prediction Examples

This notebook demonstrates ORF (Open Reading Frame) prediction using Orfipy:
1. Basic single-sequence ORF prediction
2. Custom configuration (start codons, min_len, strand filtering)
3. Batch processing with multiple sequences
4. Examining results and exporting

## Input/Output Schema

### `OrfipyInput`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| sequences | List[str] | *required* | DNA sequence(s) to analyze for open reading frames (single string or list) |
| sequence_ids | Optional[List[str]] | None | Optional sequence identifiers (defaults to seq_0, seq_1, ...) |

### `OrfipyConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| threads | int | 4 | Number of CPU threads to use |
| start_codons | str | "ATG,GTG,TTG" | Comma-separated list of start codons |
| stop_codons | str | "TAA,TAG,TGA" | Comma-separated list of stop codons |
| strand | Literal["f", "r", "b"] | "b" | Which strand(s) to scan: "f" (forward), "r" (reverse), or "b" (both) |
| min_len | int | 0 | Minimum ORF length in nucleotides |
| max_len | int | 10000 | Maximum ORF length in nucleotides |
| include_stop | bool | True | Whether to include the stop codon in the reported ORF |
| translation_table | Optional[int] | None | Optional NCBI translation table (1-33) |

### `OrfipyOutput`
| Field | Type | Description |
|-------|------|-------------|
| predicted_orfs | List[List[ORF]] | List of ORF results per input sequence |
| num_orfs | int | Total number of ORFs predicted (computed) |
| num_orfs_per_sequence | List[int] | Number of ORFs per input sequence (computed) |
| results_df | pd.DataFrame | All ORF results as a pandas DataFrame (computed) |

### `ORF`
| Field | Type | Description |
|-------|------|-------------|
| parent_id | str | Identifier of the parent/input sequence |
| orf_id | str | Unique ORF identifier within the parent sequence |
| strand | Literal["+", "-"] | Strand direction |
| frame | int | Reading frame (1, 2, or 3) |
| amino_acid_sequence | str | Translated protein sequence |
| nucleotide_sequence | str | DNA sequence of the ORF |
| amino_acid_length | int | Length of protein in amino acids |
| nucleotide_length | int | Length of ORF in nucleotides |
| nucleotide_start | int | Start position (1-indexed, inclusive) |
| nucleotide_end | int | End position (1-indexed, inclusive) |
| metrics | Dict[str, Any] | Tool-specific metrics or metadata |
| id | str | Combined identifier: parent_id + orf_id (computed) |
| gc_content | float | GC content percentage (computed) |

In [7]:
from bio_programming.bio_tools.tools.orf_prediction import (
    run_orfipy_prediction,
    OrfipyInput,
    OrfipyConfig,
)

## 1. Basic Single-Sequence ORF Prediction

Predict ORFs in a short DNA sequence using default settings.

In [8]:
# A short DNA sequence with a known ORF
sequence = "ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA"

inputs = OrfipyInput(sequences=sequence)
config = OrfipyConfig(min_len=30)

result = run_orfipy_prediction(inputs, config)
print(f"Found {result.num_orfs} ORFs")
result.results_df

Found 1 ORFs


,parent_id,orf_id,strand,frame,amino_acid_sequence,nucleotide_sequence,amino_acid_length,nucleotide_length,nucleotide_start,nucleotide_end,metrics,id,gc_content
0,seq_0,ORF.1,+,1,MVLSPADKTNVKAAWGK,ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGG...,17,54,1,54,{},seq_0_ORF.1,64.814815


## 2. Custom Configuration

Customize start codons, minimum length, and strand filtering.

In [9]:
# Only ATG start codon, forward strand only, minimum 12 nt
config_custom = OrfipyConfig(
    start_codons="ATG",
    min_len=12,
    strand="f",
    include_stop=True,
)

result_custom = run_orfipy_prediction(inputs, config_custom)
print(f"Found {result_custom.num_orfs} ORFs (forward strand, ATG only)")
result_custom.results_df

Found 1 ORFs (forward strand, ATG only)


,parent_id,orf_id,strand,frame,amino_acid_sequence,nucleotide_sequence,amino_acid_length,nucleotide_length,nucleotide_start,nucleotide_end,metrics,id,gc_content
0,seq_0,ORF.1,+,1,MVLSPADKTNVKAAWGK,ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGG...,17,54,1,54,{},seq_0_ORF.1,64.814815


## 3. Batch Processing with Multiple Sequences

Process multiple sequences at once with custom IDs.

In [10]:
sequences = [
    "ATGCCCAAATTTGGGCCCAAATTTGGGCCCAAATTTGGGTAG",
    "ATGTTTCCCGGGAAATTTCCCGGGTAA",
    "ATGAAACCCGGGAAATTTCCCGGGAAATTTCCCGGGAAATTTCCCGGGTAG",
]

inputs_batch = OrfipyInput(
    sequences=sequences,
    sequence_ids=["gene_alpha", "gene_beta", "gene_gamma"],
)
config_batch = OrfipyConfig(min_len=12)

result_batch = run_orfipy_prediction(inputs_batch, config_batch)
print(f"Total ORFs: {result_batch.num_orfs}")
print(f"ORFs per sequence: {result_batch.num_orfs_per_sequence}")
result_batch.results_df

Total ORFs: 3
ORFs per sequence: [1, 1, 1]


,parent_id,orf_id,strand,frame,amino_acid_sequence,nucleotide_sequence,amino_acid_length,nucleotide_length,nucleotide_start,nucleotide_end,metrics,id,gc_content
0,gene_alpha,ORF.1,+,1,MPKFGPKFGPKFG,ATGCCCAAATTTGGGCCCAAATTTGGGCCCAAATTTGGGTAG,13,42,1,42,{},gene_alpha_ORF.1,47.619048
1,gene_beta,ORF.1,+,1,MFPGKFPG,ATGTTTCCCGGGAAATTTCCCGGGTAA,8,27,1,27,{},gene_beta_ORF.1,48.148148
2,gene_gamma,ORF.1,+,1,MKPGKFPGKFPGKFPG,ATGAAACCCGGGAAATTTCCCGGGAAATTTCCCGGGAAATTTCCCG...,16,51,1,51,{},gene_gamma_ORF.1,50.980392


## 4. Examining Individual ORF Results

In [11]:
if result_batch.num_orfs > 0:
    orf = result_batch.predicted_orfs[0][0]
    print(f"Parent ID: {orf.parent_id}")
    print(f"ORF ID: {orf.orf_id}")
    print(f"Strand: {orf.strand}")
    print(f"Frame: {orf.frame}")
    print(f"Position: {orf.nucleotide_start}-{orf.nucleotide_end}")
    print(f"AA length: {orf.amino_acid_length}")
    print(f"NT length: {orf.nucleotide_length}")
    print(f"Protein: {orf.amino_acid_sequence}")

Parent ID: gene_alpha
ORF ID: ORF.1
Strand: +
Frame: 1
Position: 1-42
AA length: 13
NT length: 42
Protein: MPKFGPKFGPKFG


## 5. Exporting Results

In [ ]:
# Export to CSV
result_batch.export("orfipy_results", export_path="./example_output", file_format="csv")
print("Exported to ./example_output/orfipy_results.csv")

# Export protein sequences as FASTA
result_batch.export("orfipy_results", export_path="./example_output", file_format="faa")
print("Exported to ./example_output/orfipy_results.faa")